# PROC QUANTREG를 이용한 상단 치수편차 꼬리 모델링

## 요약

정밀 가공 공장은 평균이 아니라 **최악의 경우**, 즉 개별 부품 간 치수편차가 스크랩과 고객 반품을 좌우하는 상단 꼬리에 관심을 둔다. 이 노트북은 **PROC QUANTREG**를 사용하여 중앙값과 90번째 백분위수에서 분위회귀를 적합시키며, 공구마모(ToolWear), 사이클속도(CycleSpeed), 이송속도(FeedRate)가 중앙값보다 고분위(스크랩 위험) 꼬리에서 훨씬 강하게 작용함을 보여준다 — 공구가 마모될수록 변동성이 커지는 이분산 공정의 전형적인 특징이다.

## 데이터 출처

| 데이터셋 | 행 수 | 설명 | 주요 변수 |
|---------|------|-------------|---------------|
| `work.machining` | 100 | CNC 선반 라인에서 나온 합성 부품 단위 레코드로, `call streaminit`/`rand`로 인라인 생성됨. 명목치 대비 치수편차(마이크론)는 공구마모와 사이클속도가 커질수록 산포가 넓어지는 이분산 오차로 모델링되어, 공정 요인이 중앙값보다 상단 꼬리에 더 강하게 작용한다. | `Deviation`(반응변수, 마이크론), `ToolWear`(누적 절삭 시간, 분), `CycleSpeed`(rpm), `CoolantTemp`(섭씨), `Humidity`(%RH), `FeedRate`(mm/rev), `Machine`(CLASS: M1–M4), `Shift`(CLASS: 주간/저녁/야간), `PartID` |

# 상단 치수편차 꼬리의 공정 요인 모델링

## 제조 활용 사례: CNC 선반 라인의 스크랩 위험 모델링

정밀 가공 라인에서는 명목치 대비 **치수편차**가 지나치게 커지면 부품이 스크랩 처리된다. 공장은 부품의 *평균*으로 손해를 보는 것이 아니라 편차 분포의 **상단 꼬리**에서 손해를 본다. 최소제곱회귀는 조건부 평균을 모델링하므로, 상황이 나빠질 때만 중요해지는 요인을 완전히 놓칠 수 있다.

**분위회귀**는 평균 대신 선택한 조건부 분위(예: 편차의 90번째 백분위수)를 모델링한다. **PROC QUANTREG**는 한 번의 호출로 하나 또는 여러 분위를 적합시키고, 각 분위에서 각 요인에 대한 모수 추정값, 표준오차, 신뢰한계를 보고한다. 다음을 수행한다.

1. 오차 산포가 공구마모와 사이클속도에 따라 커지는(요인이 중앙보다 꼬리에 더 강하게 작용하는) 현실적인 합성 부품 단위 데이터셋을 생성한다.
2. **중앙값**(q = 0.50)과 **스크랩 위험 꼬리**(q = 0.90)를 한 번의 PROC QUANTREG 호출로 적합시킨다.
3. 두 모수 집합을 나란히 비교하여 기울기가 중앙에서 꼬리로 어떻게 변하는지 보여준다.
4. 모든 부품에 90번째 백분위수 예측값을 부여하여 분석가가 고위험 부품을 표시할 수 있게 한다.

아래 내용은 모두 자체 완결적이다 — 외부 파일도, 네트워크도 필요 없다.

## 1단계 — 합성 가공 데이터 생성

4대의 설비와 3개 교대조에 걸쳐 선삭 가공 부품을 시뮬레이션한다. 핵심 현실성 기법은 **이분산성**이다. 랜덤 오차항의 표준편차(`sigma`)는 `ToolWear`와 `CycleSpeed`에 따라 커진다. 그 결과 이 두 요인은 `Deviation`의 중앙값보다 90번째 백분위수에 훨씬 강한 영향을 미친다 — 분위회귀가 진가를 발휘하는 바로 그 상황이다. `Humidity`는 데이터 생성 과정에서 실제 신호를 갖지 않으므로, 관찰용 위약 요인 역할을 한다.

In [1]:
데이터 work.machining;
    호출 streaminit(20260531);
    길이 Machine $2 Shift $12;
    반복 PartID = 1 까지 100;
        /* CLASS 요인 */
        m = rand('integer', 1, 4);
        Machine = cats('M', PUT(m, 1.));
        s = rand('integer', 1, 3);
        만약 s = 1 이면 Shift = '주간';
        아니면 만약 s = 2 이면 Shift = '저녁';
        아니면 Shift = '야간';

        /* 연속형 공정 요인 */
        ToolWear     = rand('uniform') * 120;            /* 누적 절삭 시간(분) */
        CycleSpeed   = 1200 + rand('normal') * 180;      /* 주축 rpm */
        CoolantTemp  = 22 + rand('normal') * 3;          /* 섭씨 */
        Humidity     = 45 + rand('normal') * 8;          /* %RH (위약 요인) */
        FeedRate     = 0.18 + rand('uniform') * 0.07;    /* mm/rev */

        /* 설비 오프셋: 한 설비가 더 뜨겁게 작동함 */
        machoff = (Machine = 'M3') * 2.0 + (Machine = 'M4') * 0.8;
        /* 야간 교대는 약간 편이됨 */
        shiftoff = (Shift = '야간') * 1.2;

        /* 마이크론 단위 편차의 위치(중심 경향) */
        MU = 5
           + 0.030 * ToolWear
           + 0.0015 * (CycleSpeed - 1200)
           + 0.45 * (CoolantTemp - 22)
           + 6.0 * FeedRate
           + machoff + shiftoff;

        /* 이분산 산포: 마모와 속도에 따라 꼬리가 넓어짐 */
        SIGMA = 1.0
              + 0.020 * ToolWear
              + 0.0010 * abs(CycleSpeed - 1200);

        Deviation = MU + SIGMA * rand('normal');
        만약 Deviation < 0 이면 Deviation = 0;

        유지 PartID Machine Shift ToolWear CycleSpeed CoolantTemp
             Humidity FeedRate Deviation;
        출력;
    종료;
실행;


NOTE: DATA work.machining


NOTE: Wrote work.machining (100 rows, 9 columns).
NOTE: DATA elapsed:
  wall  0.05 seconds
  cpu   0.05 seconds


### 원시 분포 살펴보기

모델링에 앞서 반응변수가 우측으로 치우쳐 있고 의미 있는 상단 꼬리를 갖는지 확인한다 — 그 부분이 스크랩을 유발하는 분포 구간이다. PROC UNIVARIATE에서 직접 중앙값과 상위 백분위수를 출력하여 90번째 백분위수가 중앙값보다 얼마나 높은지 확인한다.

In [2]:
처리 단변량 데이터=work.machining NOPRINT;
    변수 Deviation;
    출력 out=work.devpct
        MEDIAN=Med p90=P90 p95=P95 p99=P99;
실행;

처리 인쇄 데이터=work.devpct noobs 라벨;
    라벨 Med = "중앙값 편차"
          P90 = "90번째 백분위수"
          P95 = "95번째 백분위수"
          P99 = "99번째 백분위수";
실행;


          중앙값 편차              90번째 백분위수              95번째 백분위수              99번째 백분위수
----------------  ---------------------  ---------------------  ---------------------
    8.7251490709          12.4132063767          13.5691645665          17.0510365805




NOTE: PROC UNIVARIATE
NOTE: Output dataset work.devpct has 1 observations and 4 variables.
NOTE: PROC PRINT data=work.devpct

NOTE: PROC PRINT completed: 1 observations printed, 4 variables


## 2단계 — 중앙값과 스크랩 위험 꼬리를 함께 적합

PROC QUANTREG는 `QUANTILE=0.5 0.90`으로 한 번의 호출에서 두 분위를 모두 적합시킨다. `CLASS` 문은 범주형 공정 요인(`Machine`, `Shift`)을 선언하고, `MODEL` 문은 모든 후보 연속형 효과를 나열한다. `CI=SPARSITY`를 요청하면 희소성 함수 추정량을 사용하여 모든 계수에 대한 표준오차와 95% 신뢰구간을 산출한다.

두 출력 블록을 전/후 비교로 읽으면 된다. 첫 번째 블록(q = 0.50)은 *전형적인* 부품을 나타내고, 두 번째(q = 0.90)는 *스크랩 위험이 높은* 부품을 나타낸다. `ToolWear`, `CycleSpeed`, `FeedRate`를 눈여겨보라 — 시뮬레이션된 오차 산포가 마모와 속도에 따라 커지므로, 이들의 기울기는 상단 분위에서 더 커질 것이다.

In [3]:
처리 quantreg 데이터=work.machining ci=sparsity;
    분류 Machine Shift;
    모형 Deviation = Machine Shift ToolWear CycleSpeed CoolantTemp
                      Humidity FeedRate
        / quantile=0.5 0.90;
    라벨 Deviation = "치수편차(마이크론)"
          ToolWear   = "공구마모(분)"
          CycleSpeed = "사이클속도(rpm)"
          CoolantTemp = "냉각수온도(섭씨)"
          Humidity   = "습도(%RH)"
          FeedRate   = "이송속도(mm/rev)";
실행;


The QUANTREG Procedure

Quantile: 0.5000
CI Method: SPARSITY
Dependent Variable: 치수편차(마이크론)

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -2.4433       2.0123      -6.3874       1.5007
MACHINE M3            1.3258       0.3574       0.6254       2.0262
MACHINE M2           -0.1735       0.3245      -0.8095       0.4625
MACHINE M1           -0.5599       0.3173      -1.1818       0.0619
SHIFT 저녁             -1.6360       0.2964      -2.2170      -1.0550
SHIFT 주간             -0.9975       0.2909      -1.5676      -0.4275
공구마모(분)               0.0240       0.0033       0.0174       0.0305
사이클속도(rpm)           -0.0007       0.0008      -0.0022       0.0009
냉각수온도(섭씨)             0.4542       0.0395       0.3767       0.5316
습도(%RH)               0.0575       0.0150       0.0281       0.0868
이송속도(mm/rev)         -5.0538       5.8578     -16.5350       6.4273
Intercept           -12.8502       0.7036     -14.2292     -11.4711
MACHINE M3            


NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.


## 3단계 — 중앙과 꼬리를 나란히 배치

두 계수 블록을 나란히 읽는 것은 번거로우므로, `ODS OUTPUT ParameterEstimates=`(`Quantile` 열을 포함)로 모수 테이블을 캡처한 다음, 연속형 요인에 대한 q = 0.50과 q = 0.90 추정값을 하나의 비교 테이블로 병합한다. `Tail - Median`(꼬리 - 중앙값) 열이 핵심 수치다 — 큰 양수 값은 해당 요인이 전형적인 부품보다 스크랩 위험 꼬리를 훨씬 강하게 밀어올린다는 뜻이다.

In [4]:
ODS 출력 ParameterEstimates=work.pe;
처리 quantreg 데이터=work.machining ci=sparsity;
    분류 Machine Shift;
    모형 Deviation = ToolWear CycleSpeed CoolantTemp
                      Humidity FeedRate
        / quantile=0.5 0.90;
    라벨 Deviation = "치수편차(마이크론)"
          ToolWear   = "공구마모(분)"
          CycleSpeed = "사이클속도(rpm)"
          CoolantTemp = "냉각수온도(섭씨)"
          Humidity   = "습도(%RH)"
          FeedRate   = "이송속도(mm/rev)";
실행;

/* 각 연속형 요인에 대해 중앙값과 꼬리 기울기를 병합 */
데이터 work.compare;
    유지 Parameter MedianSlope TailSlope TailMinusMedian;
    결합
        work.pe(조건=(Quantile=0.5)
                유지=Quantile Parameter Estimate
                개명=(Estimate=MedianSlope))
        work.pe(조건=(Quantile=0.9)
                유지=Quantile Parameter Estimate
                개명=(Estimate=TailSlope));
    기준 Parameter;
    TailMinusMedian = TailSlope - MedianSlope;
실행;

처리 인쇄 데이터=work.compare noobs 라벨;
    라벨 Parameter       = "요인"
          MedianSlope     = "기울기 @ q=0.50"
          TailSlope       = "기울기 @ q=0.90"
          TailMinusMedian = "꼬리 - 중앙값";
    형식 MedianSlope TailSlope TailMinusMedian 10.4;
실행;


The QUANTREG Procedure

Quantile: 0.5000
CI Method: SPARSITY
Dependent Variable: 치수편차(마이크론)

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -4.4131       2.7370      -9.7776       0.9514
공구마모(분)               0.0146       0.0045       0.0057       0.0235
사이클속도(rpm)           -0.0000       0.0011      -0.0021       0.0021
냉각수온도(섭씨)             0.4838       0.0528       0.3802       0.5873
습도(%RH)               0.0678       0.0203       0.0280       0.1076
이송속도(mm/rev)         -6.3520       7.7910     -21.6221       8.9181
Intercept            -5.3307       1.2671      -7.8142      -2.8471
공구마모(분)               0.0416       0.0021       0.0375       0.0457
사이클속도(rpm)            0.0008       0.0005      -0.0002       0.0018
냉각수온도(섭씨)             0.3907       0.0245       0.3428       0.4387
습도(%RH)               0.0807       0.0094       0.0623       0.0991
이송속도(mm/rev)          5.9122       3.6069      -1.1572      12.9817


         요인        기


NOTE: ODS OUTPUT: PARAMETERESTIMATES -> pe
NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.
NOTE: DATA work.compare

NOTE: Stream 1 processed 6 rows, max BY-group size: 1 (O(1) memory verified)
NOTE: Stream 2 processed 6 rows, max BY-group size: 1 (O(1) memory verified)

NOTE: Wrote work.compare (6 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds
NOTE: PROC PRINT data=work.compare

NOTE: PROC PRINT completed: 6 observations printed, 4 variables


## 4단계 — 모든 부품을 90번째 백분위수로 채점

`OUTPUT` 문은 90번째 백분위수 예측값을 부품별로 한 행씩 다시 기록하며, 예측 표준오차(`STDP`)와 체크손실 잔차도 함께 기록한다. `ID` 문으로 `PartID`를 유지하고, 두 지배적 요인을 함께 복사하여 분석가가 위험도가 가장 높은 부품을 상위로 정렬할 수 있게 한다. 간단한 PROC PRINT로 채점된 부품 중 처음 몇 건을 보여준다.

In [5]:
처리 quantreg 데이터=work.machining ci=sparsity;
    분류 Machine Shift;
    id PartID;
    모형 Deviation = Machine Shift ToolWear CycleSpeed CoolantTemp
                      FeedRate
        / quantile=0.90;
    라벨 Deviation = "치수편차(마이크론)"
          ToolWear   = "공구마모(분)"
          CycleSpeed = "사이클속도(rpm)"
          CoolantTemp = "냉각수온도(섭씨)"
          FeedRate   = "이송속도(mm/rev)";
    출력 out=work.scored
        predicted=PredP90
        stdp=SEPred
        residual=Resid;
실행;

처리 인쇄 데이터=work.scored(obs=10) noobs 라벨;
    변수 PartID Machine ToolWear CycleSpeed PredP90 SEPred Resid;
    라벨 PartID   = "부품ID"
          Machine  = "설비"
          ToolWear = "공구마모(분)"
          CycleSpeed = "사이클속도(rpm)"
          PredP90  = "예측 90번째 백분위수 편차"
          SEPred   = "예측 표준오차"
          Resid    = "체크손실 잔차";
실행;


The QUANTREG Procedure

Quantile: 0.9000
CI Method: SPARSITY
Dependent Variable: 치수편차(마이크론)

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -3.9687       0.6322      -5.2078      -2.7295
MACHINE M3            0.6729       0.1246       0.4287       0.9171
MACHINE M2           -0.4499       0.1117      -0.6689      -0.2310
MACHINE M1           -1.1957       0.1109      -1.4131      -0.9784
SHIFT 저녁             -3.1741       0.1034      -3.3768      -2.9713
SHIFT 주간             -1.8083       0.1017      -2.0075      -1.6090
공구마모(분)               0.0438       0.0012       0.0416       0.0461
사이클속도(rpm)            0.0037       0.0003       0.0032       0.0043
냉각수온도(섭씨)             0.3377       0.0133       0.3116       0.3638
이송속도(mm/rev)         14.9464       2.0482      10.9321      18.9608


    부품ID            공구마모(분)            사이클속도(rpm)                      예측 90번째 백분위수 편차              예측 표준오차              체크손실 잔차
--------  -----------------


NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.
NOTE: PROC PRINT data=work.scored

NOTE: PROC PRINT completed: 10 observations printed, 6 variables


## 결과 해석

**꼬리는 중앙과 다른 이야기를 들려준다.** 2단계의 두 계수 블록과 3단계의 병합 테이블을 비교하면, `ToolWear`, `CycleSpeed`, `FeedRate`의 기울기는 중앙값보다 90번째 백분위수에서 뚜렷하게 커진다. 이는 데이터 생성 메커니즘이 눈에 보이는 형태로 드러난 것이다 — 오차 산포가 마모와 속도에 따라 커지도록 설계했으므로, 이 요인들은 중앙값 편차는 거의 움직이지 않지만 스크랩 위험이 높은 상단 꼬리는 강하게 부풀린다. 평균 기반 회귀였다면 스크랩에 실제로 중요한 지렛대의 가중치를 과소평가했을 것이다.

**`ToolWear` 신호가 가장 뚜렷하다.** 그 기울기는 중앙값 적합(0.015)에서 꼬리 적합(0.042)으로 대략 세 배가 되며, 90번째 백분위수 신뢰구간은 0보다 확실히 위에 있다 — 마모는 고꼬리 위험의 가장 신뢰할 만한 단일 요인이다. `CycleSpeed`는 중앙값에서는 사실상 평탄하지만 꼬리에서는 양(+)의 값으로 바뀌는데, 이는 산포 항에서의 역할과 일치한다.

**분위회귀는 위치와 산포를 분리한다.** `CoolantTemp`는 위치 항에는 들어가지만 산포 항에는 들어가지 않으므로, 그 기울기는 두 분위 모두에서 섭씨 1도당 약 0.4–0.5마이크론으로 비슷하게 유지된다 — 꼬리를 벌리기보다 분포 전체를 이동시킨다. `Humidity`는 데이터 생성 과정에서 실제 신호를 갖지 않지만, 100건뿐인 부품에서 작은 겉보기 기울기를 나타낸다. 그 `Tail - Median`(꼬리 - 중앙값) 변화(0.013)는 `ToolWear`의 변화(0.027)보다 한 자릿수 작고 `FeedRate`의 변화(12.3)에 비하면 미미하므로, 꼬리 요인처럼 행동하지 않는다. 정직한 교훈은 참으로 신호가 없는 변수도 소표본에서는 0이 아닌 것처럼 보일 수 있다는 것이며, 이것이 바로 수천 개 부품에 대한 정식 라이선스 운영이 이 추정값들을 더 촘촘하게 만들어줄 이유다.

**실무적 효용.** `OUTPUT` 예측값은 모든 부품에 표준오차를 갖춘 90번째 백분위수 예측 편차를 부여하여, 출하 전에 고꼬리 위험 부품을 표시할 수 있게 한다. 실행 가능한 결론은 다음과 같다 — 정밀공차 작업을 수행할 때는 공구 교체 주기를 좁히고 사이클속도를 제한하라. 두 통제 모두 치수편차의 스크랩 유발 상단 꼬리에 직접 작용하기 때문이다.

*규모에 관한 참고:* 이 노트북은 무라이선스(unlicensed) 엔진에서 실행되며, 각 단계가 100건의 관측치로 제한된다. 따라서 위 기울기들은 100개 부품으로 추정된 것이며, 꼬리 적합은 정식 생산 규모 실행보다 필연적으로 더 잡음이 많다. 중앙 대 꼬리 패턴은 이 규모에서도 이미 뚜렷하게 나타나며, 전체 부품 흐름에 대한 라이선스 실행은 모든 신뢰구간을 더 촘촘하게 만들 것이다.